# 

# Web Page Analyer
Analyzes website structure and evaluates web characteristics.

## Imports

In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
from urllib.parse import urljoin, urlparse
import time
import re

In [2]:
class WebPageAnalyzer:
    def __init__(self, base_url, max_pages=30):
        self.base_url = base_url
        self.max_pages = max_pages
        self.visited_urls = set()
        self.pages_data = []
        self.session = requests.Session()
        self.session.headers.update({
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
        })
    
    def is_valid_url(self, url):
        try:
            base_domain = urlparse(self.base_url).netloc
            url_domain = urlparse(url).netloc
            return base_domain == url_domain
        except:
            return False
    
    def extract_links(self, soup, current_url):
        links = []
        for link in soup.find_all('a', href=True):
            href = link['href']
            absolute_url = urljoin(current_url, href)
            if self.is_valid_url(absolute_url):
                links.append(absolute_url)
        return links
    
    def is_dynamic_page(self, url):
        has_query_params = '?' in url
        
        dynamic_patterns = [
            r'/\d+/',       # Numbers in path
            r'\.php',       # PHP files
            r'\.asp',       # ASP files
            r'/api/',       # API endpoints
            r'/search',     # Search pages
            r'/category',   # Category pages
        ]
        
        has_dynamic_pattern = any(re.search(pattern, url, re.IGNORECASE) 
                                for pattern in dynamic_patterns)
        
        return has_query_params or has_dynamic_pattern
    
    def analyze_page(self, url):
        try:
            print(f"Analyzing {url} ...")
            response = self.session.get(url, timeout=10)
            response.raise_for_status()
            
            soup = BeautifulSoup(response.content, 'html.parser')
            
            # Extract visible text
            text_content = soup.get_text()
            content_length = len(text_content.strip())
            
            # Count outbound links
            outbound_links = self.extract_links(soup, url)
            outbound_count = len(outbound_links)
            
            # Check if page is dynamic
            is_dynamic = self.is_dynamic_page(url)
            
            page_data = {
                'URL': url,
                'Content Length (chars)': content_length,
                'Outbound Links Count': outbound_count,
                'Is Dynamic': 'Yes' if is_dynamic else 'No'
            }
            
            self.pages_data.append(page_data)
            
            # Return new URLs to crawl
            new_urls = [link for link in outbound_links 
                       if link not in self.visited_urls]
            
            return new_urls
            
        except Exception as e:
            print(f"Error analyzing {url}: {str(e)}")
            return []
    
    def crawl_website(self):
        print(f"Crawling {self.base_url} ...")
        
        urls_to_visit = [self.base_url]
        
        while urls_to_visit and len(self.visited_urls) < self.max_pages:
            current_url = urls_to_visit.pop(0)
            
            if current_url in self.visited_urls:
                continue
                
            self.visited_urls.add(current_url)
            
            # Analyze current page
            new_urls = self.analyze_page(current_url)
            
            # Add new URLs to visit queue
            for url in new_urls:
                if url not in self.visited_urls and url not in urls_to_visit:
                    urls_to_visit.append(url)
            
            # Delay between requests
            time.sleep(1)
        
        print(f"Crawl completed. Analyzed {len(self.pages_data)} pages.")
    
    def create_analysis_table(self):
        df = pd.DataFrame(self.pages_data)
        return df
    
    def calculate_statistics(self):
        if not self.pages_data:
            return None
        
        content_lengths = [page['Content Length (chars)'] for page in self.pages_data]
        link_counts = [page['Outbound Links Count'] for page in self.pages_data]
        dynamic_pages = [page['Is Dynamic'] == 'Yes' for page in self.pages_data]
        
        stats = {
            'Total Pages Analyzed': len(self.pages_data),
            'Content Length - Min': min(content_lengths),
            'Content Length - Max': max(content_lengths),
            'Content Length - Average': sum(content_lengths) / len(content_lengths),
            'Outbound Links - Min': min(link_counts),
            'Outbound Links - Max': max(link_counts),
            'Outbound Links - Average': sum(link_counts) / len(link_counts),
            'Dynamic Pages Count': sum(dynamic_pages),
            'Static Pages Count': len(dynamic_pages) - sum(dynamic_pages),
            'Dynamic Pages Percentage': (sum(dynamic_pages) / len(dynamic_pages)) * 100
        }
        
        return stats
    
    def evaluate_statements(self, stats):
        evaluations = []
        
        # Statement 1: Content is heterogeneous and without centralized design
        content_variance = stats['Content Length - Max'] - stats['Content Length - Min']
        link_variance = stats['Outbound Links - Max'] - stats['Outbound Links - Min']
        
        heterogeneous = content_variance > stats['Content Length - Average'] or \
                       link_variance > stats['Outbound Links - Average']
        
        eval1 = {
            'Statement': 'Content is heterogeneous and without centralized design',
            'Matches': heterogeneous,
            'Reasoning': f"Content length variance: {content_variance} chars, Link count variance: {link_variance}"
        }
        evaluations.append(eval1)
        
        # Statement 2: Links form the graph structure of the web
        has_links = stats['Outbound Links - Average'] > 0
        
        eval2 = {
            'Statement': 'Links form the graph structure of the web',
            'Matches': has_links,
            'Reasoning': f"Average {stats['Outbound Links - Average']:.1f} outbound links per page"
        }
        evaluations.append(eval2)
        
        # Statement 3: Pages can be created dynamically
        has_dynamic = stats['Dynamic Pages Count'] > 0
        
        eval3 = {
            'Statement': 'Pages can be created dynamically',
            'Matches': has_dynamic,
            'Reasoning': f"{stats['Dynamic Pages Count']} dynamic pages found ({stats['Dynamic Pages Percentage']:.1f}%)"
        }
        evaluations.append(eval3)
        
        return evaluations

# Initial settings

In [11]:

website_url = "https://blog.ethereum.org/"  # Target website URL
max_pages = 30  # Maximum number of pages to analyze

print(f"Starting analysis of {website_url}")
print(f"Maximum pages: {max_pages}")

Starting analysis of https://blog.ethereum.org/
Maximum pages: 30


## Create analyzer and start crawling

In [12]:
analyzer = WebPageAnalyzer(website_url, max_pages)
analyzer.crawl_website()

Crawling https://blog.ethereum.org/ ...
Analyzing https://blog.ethereum.org/ ...
Analyzing https://blog.ethereum.org/#main-content ...
Analyzing https://blog.ethereum.org/category/research-and-development ...
Analyzing https://blog.ethereum.org/category/events ...
Analyzing https://blog.ethereum.org/category/organizational ...
Analyzing https://blog.ethereum.org/category/ecosystem-support-program ...
Analyzing https://blog.ethereum.org/category/ethereum-org ...
Analyzing https://blog.ethereum.org/category/security ...
Analyzing https://blog.ethereum.org/category/next-billion ...
Analyzing https://blog.ethereum.org/category/protocol ...
Analyzing https://blog.ethereum.org/category/efi ...
Analyzing https://blog.ethereum.org/languages ...
Analyzing https://blog.ethereum.org/2025/06/04/ef-treasury-policy ...
Analyzing https://blog.ethereum.org/2025/06/03/devconnect-arg-scholars ...
Analyzing https://blog.ethereum.org/2025/06/03/checkpoint-3 ...
Analyzing https://blog.ethereum.org/2025/06/

/tmp/ipykernel_16825/1296630532.py:52: XMLParsedAsHTMLWarning: It looks like you're using an HTML parser to parse an XML document.

Assuming this really is an XML document, what you're doing might work, but you should know that using an XML parser will be more reliable. To parse this document as XML, make sure you have the Python package 'lxml' installed, and pass the keyword argument `features="xml"` into the BeautifulSoup constructor.

If you want or need to use an HTML parser on this document, you can make this warning go away by filtering it. To do that, run this code before calling the BeautifulSoup constructor:

    from bs4 import XMLParsedAsHTMLWarning
    import warnings

    warnings.filterwarnings("ignore", category=XMLParsedAsHTMLWarning)

  soup = BeautifulSoup(response.content, 'html.parser')


Analyzing https://blog.ethereum.org/archive ...
Analyzing https://blog.ethereum.org/category/research-and-development#main-content ...
Analyzing https://blog.ethereum.org/2025/04/29/checkpoint-2 ...
Analyzing https://blog.ethereum.org/2025/04/23/pectra-mainnet ...
Analyzing https://blog.ethereum.org/2025/04/10/epf-6 ...
Analyzing https://blog.ethereum.org/2025/04/10/epf-5-recap ...
Analyzing https://blog.ethereum.org/2025/03/25/acdcheckpoint-001 ...
Analyzing https://blog.ethereum.org/2025/03/18/hoodi-holesky ...
Crawl completed. Analyzed 30 pages.


## Create results table

In [13]:
df = analyzer.create_analysis_table()
print("Page Analysis Table:")
print("="*80)
display(df)

Page Analysis Table:


,URL,Content Length (chars),Outbound Links Count,Is Dynamic
0,https://blog.ethereum.org/,6766,53,No
1,https://blog.ethereum.org/#main-content,6766,53,No
2,https://blog.ethereum.org/category/research-an...,5646,50,Yes
3,https://blog.ethereum.org/category/events,5493,50,Yes
4,https://blog.ethereum.org/category/organizational,5723,50,Yes
5,https://blog.ethereum.org/category/ecosystem-s...,3310,50,Yes
6,https://blog.ethereum.org/category/ethereum-org,4803,50,Yes
7,https://blog.ethereum.org/category/security,5342,50,Yes
8,https://blog.ethereum.org/category/next-billion,6547,50,Yes
9,https://blog.ethereum.org/category/protocol,6785,50,Yes


## Calculate statistics

In [14]:
stats = analyzer.calculate_statistics()
print("\nOverall Statistics:")
print("="*40)
for key, value in stats.items():
    if isinstance(value, float):
        print(f"{key}: {value:.2f}")
    else:
        print(f"{key}: {value}")


Overall Statistics:
Total Pages Analyzed: 30
Content Length - Min: 1407
Content Length - Max: 294467
Content Length - Average: 16973.47
Outbound Links - Min: 0
Outbound Links - Max: 1156
Outbound Links - Average: 76.87
Dynamic Pages Count: 25
Static Pages Count: 5
Dynamic Pages Percentage: 83.33


## Evaluate statements

In [15]:
evaluations = analyzer.evaluate_statements(stats)
print("\nStatement Evaluation:")
print("="*50)

for eval_item in evaluations:
    status = "✓ MATCHES" if eval_item['Matches'] else "✗ DOESN'T MATCH"
    print(f"\nStatement: {eval_item['Statement']}")
    print(f"Result: {status}")
    print(f"Reasoning: {eval_item['Reasoning']}")


Statement Evaluation:

Statement: Content is heterogeneous and without centralized design
Result: ✓ MATCHES
Reasoning: Content length variance: 293060 chars, Link count variance: 1156

Statement: Links form the graph structure of the web
Result: ✓ MATCHES
Reasoning: Average 76.9 outbound links per page

Statement: Pages can be created dynamically
Result: ✓ MATCHES
Reasoning: 25 dynamic pages found (83.3%)


## Save results to CSV file

In [16]:
df.to_csv('website_analysis.csv', index=False)
print(f"\nResults saved to 'website_analysis.csv'")


Results saved to 'website_analysis.csv'
